# testLeague — loading a season through the ESPN adapter

Loads a season with `EspnLeagueRepository` and looks at the resulting domain objects
(`League`, `Team`, `Matchup`, `Lineup`), plus a few sanity checks against ESPN's own numbers.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import Counter

import pandas as pd

from fantasy_analysis.config import settings
from fantasy_analysis.infrastructure.espn import EspnLeagueRepository

pd.set_option("display.max_columns", 50)

SEASON = 2026


def result_for(m, team) -> str:
    """W / L / T / BYE for one team in a matchup."""
    if m.away is None:
        return "BYE"
    if m.winner is None:
        return "T"
    return "W" if m.winner == team else "L"


## Load the season

Makes several ESPN requests per week, so a full season takes a little while.

In [ ]:
repo = EspnLeagueRepository.from_settings(settings)
league = repo.get_league(SEASON)

weeks = sorted({m.week for m in league.matchups})
print(f"{league.name} ({league.season}): {league.num_teams} teams, weeks {weeks[0]}-{weeks[-1]}")
print("matchups by kind:", dict(Counter(m.kind for m in league.matchups)))

## Teams

In [ ]:
pd.DataFrame(
    [(t.team_id, t.name, ", ".join(t.owners), len(t.lineups)) for t in league.teams],
    columns=["team_id", "name", "owners", "weeks_with_lineup"],
)

## Matchups

In [ ]:
matchups_df = pd.DataFrame(
    [
        {
            "week": m.week,
            "kind": m.kind,
            "home": m.home.name,
            "home_score": m.home_score,
            "away": m.away.name if m.away else "BYE",
            "away_score": m.away_score,
            "winner": m.winner.name if m.winner else None,
        }
        for m in league.matchups
    ]
)
matchups_df

## Regular-season standings (computed from matchups)

In [ ]:
rows = []
for m in league.matchups:
    if m.kind != "REGULAR_SEASON" or m.away is None or m.away_score is None:
        continue
    sides = ((m.home, m.home_score, m.away_score), (m.away, m.away_score, m.home_score))
    for team, pf, pa in sides:
        result = result_for(m, team)
        rows.append({"team": team.name, "result": result, "points_for": pf, "points_against": pa})

results = pd.DataFrame(rows)
standings = results.groupby("team").agg(
    wins=("result", lambda r: (r == "W").sum()),
    losses=("result", lambda r: (r == "L").sum()),
    ties=("result", lambda r: (r == "T").sum()),
    points_for=("points_for", "sum"),
    points_against=("points_against", "sum"),
)
standings.sort_values(["wins", "points_for"], ascending=False)

## One team's week: lineup with actual vs. projected points

In [ ]:
weeks

In [ ]:
team = league.teams[9]
WEEK = 7
lineup = team.lineups[WEEK]

print(f"{team.name}, week {WEEK}: {lineup.points:.2f} points from starters")
pd.DataFrame(
    [
        {
            "slot": e.slot,
            "player": e.player.name,
            "position": e.player.position,
            "pro_team": e.player.pro_team,
            "points": e.points,
            "projected": e.projected_points,
            "starter": e.is_starter,
        }
        for e in lineup.entries
    ]
)

In [ ]:
# That team's schedule
pd.DataFrame(
    [
        {
            "week": m.week,
            "opponent": (m.away if m.home == team else m.home).name if m.away else "BYE",
            "result": result_for(m, team),
        }
        for m in league.schedule(team)
    ]
)

## Every lineup entry, season-long

One row per player per team per week — the main table for analysis.

In [ ]:
entries_df = pd.DataFrame(
    [
        {
            "week": week,
            "team": t.name,
            "slot": e.slot,
            "starter": e.is_starter,
            "player_id": e.player.player_id,
            "player": e.player.name,
            "position": e.player.position,
            "pro_team": e.player.pro_team,
            "points": e.points,
            "projected": e.projected_points,
        }
        for t in league.teams
        for week, lu in t.lineups.items()
        for e in lu.entries
    ]
)
print(entries_df.shape)
entries_df.head()

In [ ]:
# Points left on the bench, per team
bench = entries_df[entries_df["slot"] == "BENCH"]
bench.groupby("team")["points"].sum().sort_values(ascending=False)

## Sanity checks

In [ ]:
# ESPN's matchup score should equal the sum of starters' points
mismatches = [
    (m.week, side.name, score, round(side.lineups[m.week].points, 2))
    for m in league.matchups
    for side, score in ((m.home, m.home_score), (m.away, m.away_score))
    if side is not None and score is not None and abs(score - side.lineups[m.week].points) > 0.01
]
print("score mismatches:", mismatches or "none")

# Every team should have a lineup in every week it played
missing = [
    (m.week, side.name)
    for m in league.matchups
    for side in (m.home, m.away)
    if side is not None and m.week not in side.lineups
]
print("missing lineups:", missing or "none")

# Lineups with an empty starting slot (bench > 6)
short = entries_df[entries_df["starter"]].groupby(["week", "team"]).size()
print("lineups with an empty starting slot:")
short[short < 10]